[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/05_attention.ipynb)

# 🔴 困难：Softmax Attention

实现 Transformer 中使用的核心注意力机制。

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### 函数签名
```python
def scaled_dot_product_attention(
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
) -> torch.Tensor:   # (batch, seq_q, d_v)
    ...
```

### 规则
- **禁止** 使用 `F.scaled_dot_product_attention`
- **允许** 使用 `torch.softmax` 和 `torch.bmm`
- 必须支持自动求导（autograd）
- 必须支持交叉注意力（cross-attention），即 seq_q ≠ seq_k

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import math

/Users/xiaohui/Documents/project/TorchCode/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


1. 获取缩放因子 d_k
Q 的最后一个维度即为 d_k
2. 计算得分 (Scores): Q @ K^T
使用 torch.transpose 将 K 的最后两个维度转置: (batch, d_k, seq_k)
bmm 处理 batch 矩阵乘法: (batch, seq_q, d_k) @ (batch, d_k, seq_k) -> (batch, seq_q, seq_k)
3. 缩放得分
4. 应用 Softmax 得到权重 (Attention Weights)
在最后一个维度 (seq_k) 上进行归一化，使得每个 query 对所有 key 的权重和为 1
5. 加权求和 (Weights @ V)
(batch, seq_q, seq_k) @ (batch, seq_k, d_v) -> (batch, seq_q, d_v)


| 特性         | torch.bmm                                      | torch.matmul (@ 运算符)                                      |
|--------------|------------------------------------------------|-------------------------------------------------------------|
| **维度限制** | 严格要求 **3 维** 张量                         | 非常灵活，支持 **1D ~ ND**（任意维度）                      |
| **广播机制** | **不支持广播**<br>batch size 不一致时会报错    | **支持广播**<br>可自动对齐维度（如 3D 与 2D 相乘）          |
| **用法建议** | 输入确定为三维时使用<br>（可读性高，防止意外广播） | 推荐在通用模型代码中使用<br>（线性层、注意力机制等）        |


x.transpose(-2, -1) 和 x.transpose(-1, -2) 完全一致

In [ ]:
# ✏️ 在此处实现你的代码

def softmax(x: torch.tensor, dim:int = -1):
    max_val = torch.max(x, dim=dim, keepdim=True)[0]    
    x_exp = torch.exp(x - max_val)
    partition_sum = torch.sum(x_exp, dim=dim, keepdim=True)
    return x_exp / partition_sum

def scaled_dot_product_attention(
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
) -> torch.Tensor:   # (batch, seq_q, d_v)
    
    d_k = Q.size(-1)
    scores = torch.bmm(Q, K.transpose(-2, -1))
    scaled_scores = scores / math.sqrt(d_k)
    attn_weights = softmax(scaled_scores, -1)
    
    output = torch.bmm(attn_weights, V)
    
    return output

In [ ]:
# 🧪 调试测试

torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("输出形状：", out.shape)          # 应为 (2, 4, 8)
print("包含 NaN？    ", torch.isnan(out).any().item())  # 应为 False
print("包含 Inf？    ", torch.isinf(out).any().item())  # 应为 False

# 交叉注意力测试：seq_q ≠ seq_k
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("交叉注意力形状：", out2.shape)     # 应为 (1, 3, 32)

In [ ]:
# ✅ 提交

from torch_judge import check
check("attention")